# Tratamento — DATASUS (Bronze -> Silver)

Lê três dos quatro Parquet da camada Bronze ([extracao_bronze_datasus.ipynb](extracao_bronze_datasus.ipynb)) — `aih_aprovadas`, `valor_total` (hospitalar) e `qtd_aprovada` (ambulatorial) — e aplica as transformações definidas para a Silver, com o escopo já combinado de usar **só 2018-2024**:

1. **Deduplicação** — proteção estrutural, mesmo padrão das demais Silver.
2. **Filtro de período (2018-2024)** — antes de tratar nulos, porque o período pré-2013 tinha uma taxa de nulos de outra natureza (99%+, ver [exploracao_bronze_datasus.ipynb](exploracao_bronze_datasus.ipynb), seção 4); filtrando primeiro, a decisão de remover nulos passa a valer só para o patamar "normal" (2-4%) que sobra na janela usada.
3. **Remoção de nulos em `quantidade`** — nulo aqui é "sem registro no DATASUS", não zero; a linha é descartada, não imputada.
4. **Separação de metadados** — `arquivo_origem`/`categoria`/`metrica`/`titulo`/`descricao_variavel`/`periodo_relatorio` saem das linhas de fato e viram uma tabela `metadados`, no mesmo padrão já usado na Silver do IBGE.
5. **Junção `aih_aprovadas` + `valor_total`** numa única tabela `hospitalar` (os dois vêm do mesmo sistema, SIH/SUS, e cobrem exatamente os mesmos procedimentos — achado confirmado na EDA, seção 7).
6. **`valor_amb` não entra na Silver** — só a métrica de contagem (`qtd_aprovada`) do lado ambulatorial é usada.
7. **Reforço da semântica da coluna de valor** — nomes de coluna e `description` do schema deixam explícito que os valores em reais aqui são **totais somáveis**, ao contrário da `renda` do IBGE (que era uma média — ver [tratamento_silver_ibge.ipynb](tratamento_silver_ibge.ipynb)).
8. **Validação de schema (`pandera`)**, uma por tabela, ao final.

## 1. Imports e configuração

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from minio import Minio
from pandera.pandas import Column, Check, DataFrameSchema

In [2]:
load_dotenv(Path.cwd().parent / ".env")

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "localhost:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")

BUCKET_BRONZE = os.getenv("BUCKET_BRONZE", "bronze")
BRONZE_PREFIX = "dados_DATASUS/"

BUCKET_SILVER = os.getenv("BUCKET_SILVER", "silver")
SILVER_PREFIX = "dados_DATASUS/"

SILVER_DIR = Path.cwd().parent / "dados_processados" / "silver" / "dados_DATASUS"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

ANO_INICIO, ANO_FIM = 2018, 2024

client = Minio(MINIO_ENDPOINT, access_key=MINIO_ACCESS_KEY, secret_key=MINIO_SECRET_KEY, secure=False)
if not client.bucket_exists(BUCKET_SILVER):
    client.make_bucket(BUCKET_SILVER)
    print(f"Bucket '{BUCKET_SILVER}' criado.")

print(f"MinIO: {MINIO_ENDPOINT} | bucket bronze: {BUCKET_BRONZE} | bucket silver: {BUCKET_SILVER}")
print(f"Escopo: {ANO_INICIO}-{ANO_FIM} | Saida parquet local: {SILVER_DIR}")

MinIO: localhost:9000 | bucket bronze: bronze | bucket silver: silver
Escopo: 2018-2024 | Saida parquet local: C:\Projeto_AI\dados_processados\silver\dados_DATASUS


## 2. Carregamento da camada Bronze

Só três das quatro tabelas da Bronze são lidas — `valor_aprovado_producao_ambulatorial` (`valor_amb`) fica de fora por decisão explícita: **não vai para a Silver**. Não é uma exclusão por problema de qualidade, é um corte de escopo (só a métrica de contagem do lado ambulatorial é necessária).

In [3]:
def ler_parquet_bronze(nome_arquivo):
    data = client.get_object(BUCKET_BRONZE, f"{BRONZE_PREFIX}{nome_arquivo}").read()
    return pd.read_parquet(pd.io.common.BytesIO(data))


aih = ler_parquet_bronze("aih_aprovadas_procedimentos_hospitalares.parquet")
valor_hosp = ler_parquet_bronze("valor_total_procedimentos_hospitalares.parquet")
qtd_amb = ler_parquet_bronze("qtd_aprovada_producao_ambulatorial.parquet")

for nome, df in [("aih", aih), ("valor_hosp", valor_hosp), ("qtd_amb", qtd_amb)]:
    print(f"{nome}: {df.shape}")

aih: (32094, 11)
valor_hosp: (32094, 11)
qtd_amb: (33876, 11)


## 3. Etapa 1 — Deduplicação

Mesma proteção estrutural já aplicada em todas as Silver deste projeto: custo zero hoje (a EDA não encontrou duplicatas), defesa contra um reprocessamento futuro que infle silenciosamente as somas.

In [4]:
for nome, df in [("aih", aih), ("valor_hosp", valor_hosp), ("qtd_amb", qtd_amb)]:
    antes = len(df)
    df.drop_duplicates(inplace=True)
    print(f"{nome}: antes={antes} | depois={len(df)} | removidas={antes - len(df)}")

aih: antes=32094 | depois=32094 | removidas=0
valor_hosp: antes=32094 | depois=32094 | removidas=0
qtd_amb: antes=33876 | depois=33876 | removidas=0


## 4. Etapa 2 — Filtro de período (2018-2024)

Aplicado **antes** da remoção de nulos, de propósito: a EDA mostrou que os anos anteriores a 2013 têm 99%+ de nulos (a classificação de procedimentos mal existia nesse período) contra 2-4% de 2014 em diante. Filtrando o período primeiro, a decisão da próxima etapa ("remover nulo") passa a se aplicar só ao patamar normal de nulo — sem isso, remover nulos sobre a série inteira estaria descartando, na prática, quase todo o histórico 2007-2012 misturado com o descarte legítimo de 2018-2024.

In [5]:
for nome, df in [("aih", aih), ("valor_hosp", valor_hosp), ("qtd_amb", qtd_amb)]:
    antes = len(df)
    mask = df["ano"].between(ANO_INICIO, ANO_FIM)
    df.drop(index=df.index[~mask], inplace=True)
    print(f"{nome}: antes={antes} | depois={len(df)}")

aih: antes=32094 | depois=12481
valor_hosp: antes=32094 | depois=12481
qtd_amb: antes=33876 | depois=19761


## 5. Etapa 3 — Remoção de nulos em `quantidade`

Dentro da janela 2018-2024 o nulo volta a significar o mesmo que nas demais Silver deste projeto: "sem registro para aquele procedimento naquele ano", não um zero real. Mantê-lo traria o mesmo problema já resolvido no ISAPS e no IBGE — por isso a linha é descartada, não imputada com `0`.

In [6]:
for nome, df in [("aih", aih), ("valor_hosp", valor_hosp), ("qtd_amb", qtd_amb)]:
    antes = len(df)
    df.dropna(subset=["quantidade"], inplace=True)
    print(f"{nome}: antes={antes} | depois={len(df)} | nulos_removidos={antes - len(df)}")

aih: antes=12481 | depois=12174 | nulos_removidos=307


valor_hosp: antes=12481 | depois=12174 | nulos_removidos=307
qtd_amb: antes=19761 | depois=17775 | nulos_removidos=1986


## 6. Etapa 4 — Separação de metadados

`arquivo_origem`, `categoria`, `metrica`, `titulo`, `descricao_variavel` e `periodo_relatorio` são idênticos em todas as linhas de cada tabela de origem — mesmo achado e mesmo tratamento já aplicado na Silver do IBGE ([tratamento_silver_ibge.ipynb](tratamento_silver_ibge.ipynb), etapa 2): saem das linhas de fato e viram uma tabela `metadados`, com uma linha por métrica de origem (`aih_aprovadas`, `valor_total`, `qtd_aprovada` — `valor_amb` não entra, pela mesma decisão da etapa 2).

Duas colunas extras entram aqui, que não existiam no IBGE: `unidade` (a unidade de cada métrica: `AIH`, `Reais`, `procedimentos`) e `tipo_valor` (`contagem` ou `valor_monetario_total`) — não dá para extrair a unidade de dentro do texto de `descricao_variavel` por regex como foi feito no IBGE (o texto aqui não segue o padrão `"... (unidade)"`), então essas duas colunas são preenchidas de forma explícita por métrica.

In [7]:
METRICAS = {
    "aih_aprovadas": {"df": aih, "unidade": "AIH", "tipo_valor": "contagem"},
    "valor_total": {"df": valor_hosp, "unidade": "Reais", "tipo_valor": "valor_monetario_total"},
    "qtd_aprovada": {"df": qtd_amb, "unidade": "procedimentos", "tipo_valor": "contagem"},
}

COLS_METADADOS = ["arquivo_origem", "categoria", "metrica", "titulo", "descricao_variavel", "periodo_relatorio"]

linhas_metadados = []
for nome_metrica, cfg in METRICAS.items():
    df = cfg["df"]
    linha = df[COLS_METADADOS].iloc[0].to_dict()
    linha["unidade"] = cfg["unidade"]
    linha["tipo_valor"] = cfg["tipo_valor"]
    linhas_metadados.append(linha)

    cfg["df"] = df.drop(columns=COLS_METADADOS)

metadados = pd.DataFrame(linhas_metadados)
aih, valor_hosp, qtd_amb = METRICAS["aih_aprovadas"]["df"], METRICAS["valor_total"]["df"], METRICAS["qtd_aprovada"]["df"]
metadados

,arquivo_origem,categoria,metrica,titulo,descricao_variavel,periodo_relatorio,unidade,tipo_valor
0,AIH-aprovadas_Procedimentos-hospitalares.csv,hospitalar,aih_aprovadas,Procedimentos hospitalares do SUS - por local ...,AIH aprovadas por Procedimento e Ano atendimento,Período:2014-2024,AIH,contagem
1,Valor-total_Procedimentos-hospitalares.csv,hospitalar,valor_total,Procedimentos hospitalares do SUS - por local ...,Valor total por Procedimento e Ano atendimento,Período:2014-2024,Reais,valor_monetario_total
2,Qtd.aprovada_Producao-Ambulatorial.csv,ambulatorial,qtd_aprovada,Produção Ambulatorial do SUS - Brasil - por lo...,Qtd.aprovada por Procedimento e Ano atendimento,Período:2014-2024,procedimentos,contagem


## 7. Etapa 5 — Junção `aih_aprovadas` + `valor_total` -> tabela `hospitalar`

`aih` e `valor_hosp` cobrem exatamente o mesmo conjunto de procedimentos e o mesmo período (achado confirmado na EDA, seção 7: `diferenca=0` entre os dois conjuntos de `procedimento_codigo`) — são duas métricas do mesmo evento (uma AIH aprovada), não duas populações diferentes. Faz sentido virarem colunas de uma mesma linha (`procedimento_codigo`, `procedimento_nome`, `ano`) em vez de tabelas separadas.

**Por que a junção precisa de dois passos, e não um `merge` direto:** o instinto seria chavear por `(procedimento_codigo, ano)` — só que a linha agregada `"Total"` tem `procedimento_codigo` nulo nas duas tabelas, e usar nulo como parte da chave é frágil. Trocar para `(procedimento_nome, ano)` pareceria resolver isso (o literal `"Total"` funciona como chave sem ambiguidade)... mas a checagem abaixo mostra que `procedimento_nome` **não é uma chave segura por conta própria**: existe pelo menos um caso real (`"TRATAMENTO CONSERVADOR DE TUMOR DO SISTEMA NERVOSO CENTRAL"`) em que **dois códigos SUS diferentes têm exatamente a mesma descrição** — juntar só por nome misturaria as quantidades desses dois procedimentos distintos como se fossem um só, um erro silencioso e difícil de notar depois.

A solução é juntar em duas partes com a chave certa para cada uma: as linhas reais usam `(procedimento_codigo, ano)` (código é sempre único quando existe); a única linha por ano sem código (`"Total"`) usa `(procedimento_nome, ano)`, caso em que "Total" *é* uma chave segura (não há outra linha nula disputando o mesmo `ano`).

In [8]:
nomes_repetidos = aih[aih['procedimento_codigo'].notnull()].groupby('procedimento_nome')['procedimento_codigo'].nunique()
nomes_repetidos = nomes_repetidos[nomes_repetidos > 1]
print(f"Nomes de procedimento com mais de um codigo distinto: {len(nomes_repetidos)}")
nomes_repetidos

Nomes de procedimento com mais de um codigo distinto: 1


procedimento_nome
TRATAMENTO CONSERVADOR DE TUMOR DO SISTEMA NERVOSO CENTRAL    2
Name: procedimento_codigo, dtype: int64

In [9]:
def juntar_hospitalar(aih_df, valor_df):
    aih_df = aih_df.rename(columns={"quantidade": "quantidade_aih"})
    valor_df = valor_df.rename(columns={"quantidade": "valor_total_reais"})

    com_codigo = pd.merge(
        aih_df[aih_df["procedimento_codigo"].notnull()],
        valor_df[valor_df["procedimento_codigo"].notnull()][["procedimento_codigo", "ano", "valor_total_reais"]],
        on=["procedimento_codigo", "ano"],
        how="inner",
    )

    linha_total = pd.merge(
        aih_df[aih_df["procedimento_codigo"].isnull()],
        valor_df[valor_df["procedimento_codigo"].isnull()][["procedimento_nome", "ano", "valor_total_reais"]],
        on=["procedimento_nome", "ano"],
        how="inner",
    )

    return pd.concat([com_codigo, linha_total], ignore_index=True)


hospitalar = juntar_hospitalar(aih, valor_hosp)
print(f"aih: {len(aih)} | valor_hosp: {len(valor_hosp)} | hospitalar (apos merge): {len(hospitalar)}")
hospitalar.head()

aih: 12174 | valor_hosp: 12174 | hospitalar (apos merge): 12174


,procedimento_original,procedimento_codigo,procedimento_nome,ano,quantidade_aih,valor_total_reais
0,0201010038 BIOPSIA CIRURGICA DE TIREOIDE,0201010038,BIOPSIA CIRURGICA DE TIREOIDE,2018,27.0,11129.92
1,0201010038 BIOPSIA CIRURGICA DE TIREOIDE,0201010038,BIOPSIA CIRURGICA DE TIREOIDE,2019,81.0,42359.42
2,0201010038 BIOPSIA CIRURGICA DE TIREOIDE,0201010038,BIOPSIA CIRURGICA DE TIREOIDE,2020,35.0,21422.33
3,0201010038 BIOPSIA CIRURGICA DE TIREOIDE,0201010038,BIOPSIA CIRURGICA DE TIREOIDE,2021,60.0,24336.04
4,0201010038 BIOPSIA CIRURGICA DE TIREOIDE,0201010038,BIOPSIA CIRURGICA DE TIREOIDE,2022,162.0,77472.40


## 8. Etapa 6 — Reforço da semântica da coluna de valor

`ambulatorial` (o novo nome para o antigo `qtd_amb`, já sem as colunas de metadados) e `quantidade_aih` em `hospitalar` são **contagens** — número de procedimentos/AIH aprovados, uma unidade discreta, sempre somável entre procedimentos ou anos. `valor_total_reais` é um **valor monetário total** — também somável (é uma soma, não uma média), mas por um motivo diferente do da contagem: soma de reais aprovados continua sendo reais aprovados, então tanto faz agregar por procedimento, por ano ou por ambos.

Isso é o oposto exato da `renda` do IBGE, cujo valor era uma **média** mensal e não podia ser somado (ver [tratamento_silver_ibge.ipynb](tratamento_silver_ibge.ipynb), etapa 3/5). O nome de cada coluna já carrega essa informação (`quantidade_*` = contagem, `valor_total_reais` = monetário somável), e a etapa seguinte deixa isso também no `description` de cada schema — documentado em dois lugares, não só um, para quem for consumir a Silver não precisar adivinhar a partir do nome.

In [10]:
ambulatorial = qtd_amb.rename(columns={"quantidade": "quantidade_aprovada"})
ambulatorial.head()

,procedimento_original,procedimento_codigo,procedimento_nome,ano,quantidade_aprovada
5,0101010010 ATIVIDADE EDUCATIVA / ORIENTACAO EM...,0101010010,ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...,2018,14084276.0
6,0101010010 ATIVIDADE EDUCATIVA / ORIENTACAO EM...,0101010010,ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...,2019,11388152.0
7,0101010010 ATIVIDADE EDUCATIVA / ORIENTACAO EM...,0101010010,ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...,2020,7221476.0
8,0101010010 ATIVIDADE EDUCATIVA / ORIENTACAO EM...,0101010010,ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...,2021,6569587.0
9,0101010010 ATIVIDADE EDUCATIVA / ORIENTACAO EM...,0101010010,ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...,2022,8399094.0


## 9. Etapa 7 — Validação de schema (`pandera`), uma por tabela

`hospitalar` reúne duas métricas (por isso não carrega uma coluna `metrica` única, ao contrário do padrão do IBGE — ver etapa 6); `ambulatorial` continua sendo uma métrica só, então mantém rastreabilidade simples com `metadados` pelo próprio nome (`qtd_aprovada`) sem precisar de coluna extra. Cada schema documenta no `description` da coluna de valor se ela é somável ou não — aqui, as duas são, o que fica dito explicitamente para não virar uma suposição não verificada de quem for reaproveitar os dados.

In [11]:
schema_metadados = DataFrameSchema(
    {
        "arquivo_origem": Column(str, nullable=False),
        "categoria": Column(str, Check.isin(["hospitalar", "ambulatorial"]), nullable=False),
        "metrica": Column(str, nullable=False, unique=True),
        "titulo": Column(str, nullable=False),
        "descricao_variavel": Column(str, nullable=False),
        "periodo_relatorio": Column(str, nullable=False),
        "unidade": Column(str, nullable=False),
        "tipo_valor": Column(str, Check.isin(["contagem", "valor_monetario_total"]), nullable=False),
    },
    coerce=True,
    strict=True,
)

schema_hospitalar = DataFrameSchema(
    {
        "procedimento_codigo": Column(str, nullable=True),  # nulo so na linha agregada "Total"
        "procedimento_original": Column(str, nullable=False),
        "procedimento_nome": Column(str, nullable=False),
        "ano": Column(int, Check.in_range(2018, 2024), nullable=False),
        "quantidade_aih": Column(
            float, Check.ge(0), nullable=False,
            description="Contagem de AIH aprovadas. Somavel entre procedimentos e/ou anos.",
        ),
        "valor_total_reais": Column(
            float, Check.ge(0), nullable=False,
            description=(
                "Valor monetario TOTAL (reais), nao uma media - diferente da renda do IBGE. "
                "Somavel entre procedimentos e/ou anos."
            ),
        ),
    },
    coerce=True,
    strict=True,
)

schema_ambulatorial = DataFrameSchema(
    {
        "procedimento_codigo": Column(str, nullable=True),  # nulo so na linha agregada "Total"
        "procedimento_original": Column(str, nullable=False),
        "procedimento_nome": Column(str, nullable=False),
        "ano": Column(int, Check.in_range(2018, 2024), nullable=False),
        "quantidade_aprovada": Column(
            float, Check.ge(0), nullable=False,
            description="Contagem de procedimentos ambulatoriais aprovados. Somavel entre procedimentos e/ou anos.",
        ),
    },
    coerce=True,
    strict=True,
)

metadados = schema_metadados.validate(metadados)
hospitalar = schema_hospitalar.validate(hospitalar)
ambulatorial = schema_ambulatorial.validate(ambulatorial)
print("Todos os schemas validos.")

Todos os schemas validos.


## 10. Gravação em Parquet (camada Silver)

Três arquivos — `metadados`, `hospitalar`, `ambulatorial` — gravados localmente em `dados_processados/silver/dados_DATASUS/` e enviados ao MinIO no bucket `silver`, prefixo `dados_DATASUS/`.

In [12]:
tabelas_finais = {"metadados": metadados, "hospitalar": hospitalar, "ambulatorial": ambulatorial}

arquivos_gerados = {}
for nome, df in tabelas_finais.items():
    out_path = SILVER_DIR / f"{nome}.parquet"
    df.to_parquet(out_path, engine="pyarrow", index=False)
    arquivos_gerados[nome] = out_path

    object_name = f"{SILVER_PREFIX}{nome}.parquet"
    client.fput_object(BUCKET_SILVER, object_name, str(out_path))
    print(f"Gravado: {out_path} ({len(df)} linhas) -> s3://{BUCKET_SILVER}/{object_name}")

Gravado: C:\Projeto_AI\dados_processados\silver\dados_DATASUS\metadados.parquet (3 linhas) -> s3://silver/dados_DATASUS/metadados.parquet


Gravado: C:\Projeto_AI\dados_processados\silver\dados_DATASUS\hospitalar.parquet (12174 linhas) -> s3://silver/dados_DATASUS/hospitalar.parquet
Gravado: C:\Projeto_AI\dados_processados\silver\dados_DATASUS\ambulatorial.parquet (17775 linhas) -> s3://silver/dados_DATASUS/ambulatorial.parquet


## 11. Conferência final

In [13]:
for nome, out_path in arquivos_gerados.items():
    df = pd.read_parquet(out_path)
    print(f"{nome}: {df.shape} | colunas: {list(df.columns)}")

metadados: (3, 8) | colunas: ['arquivo_origem', 'categoria', 'metrica', 'titulo', 'descricao_variavel', 'periodo_relatorio', 'unidade', 'tipo_valor']
hospitalar: (12174, 6) | colunas: ['procedimento_original', 'procedimento_codigo', 'procedimento_nome', 'ano', 'quantidade_aih', 'valor_total_reais']
ambulatorial: (17775, 5) | colunas: ['procedimento_original', 'procedimento_codigo', 'procedimento_nome', 'ano', 'quantidade_aprovada']


In [14]:
pd.read_parquet(arquivos_gerados['metadados'])

,arquivo_origem,categoria,metrica,titulo,descricao_variavel,periodo_relatorio,unidade,tipo_valor
0,AIH-aprovadas_Procedimentos-hospitalares.csv,hospitalar,aih_aprovadas,Procedimentos hospitalares do SUS - por local ...,AIH aprovadas por Procedimento e Ano atendimento,Período:2014-2024,AIH,contagem
1,Valor-total_Procedimentos-hospitalares.csv,hospitalar,valor_total,Procedimentos hospitalares do SUS - por local ...,Valor total por Procedimento e Ano atendimento,Período:2014-2024,Reais,valor_monetario_total
2,Qtd.aprovada_Producao-Ambulatorial.csv,ambulatorial,qtd_aprovada,Produção Ambulatorial do SUS - Brasil - por lo...,Qtd.aprovada por Procedimento e Ano atendimento,Período:2014-2024,procedimentos,contagem


In [15]:
pd.read_parquet(arquivos_gerados['hospitalar']).sort_values(['ano', 'valor_total_reais'], ascending=[True, False]).head(10)

,procedimento_original,procedimento_codigo,procedimento_nome,ano,quantidade_aih,valor_total_reais
12167,Total,NaN,Total,2018,11981372.0,1.510300e+10
209,0303010037 TRATAMENTO DE OUTRAS DOENCAS BACTER...,0303010037,TRATAMENTO DE OUTRAS DOENCAS BACTERIANAS,2018,294444.0,8.493189e+08
10964,0415010012 TRATAMENTO C/ CIRURGIAS MULTIPLAS,0415010012,TRATAMENTO C/ CIRURGIAS MULTIPLAS,2018,249120.0,7.727357e+08
1248,0303140151 TRATAMENTO DE PNEUMONIAS OU INFLUEN...,0303140151,TRATAMENTO DE PNEUMONIAS OU INFLUENZA (GRIPE),2018,635491.0,6.513566e+08
1685,0310010039 PARTO NORMAL,0310010039,PARTO NORMAL,2018,1006664.0,5.603424e+08
10240,0411010034 OPERACAO CESARIANA,0411010034,OPERACAO CESARIANA,2018,683078.0,4.932856e+08
10992,0415020050 PROCEDIMENTOS SEQUENCIAIS EM ONCOLOGIA,0415020050,PROCEDIMENTOS SEQUENCIAIS EM ONCOLOGIA,2018,55598.0,4.227626e+08
1332,0303160063 TRATAMENTO DE TRANSTORNOS RESPIRATO...,0303160063,TRATAMENTO DE TRANSTORNOS RESPIRATORIOS E CARD...,2018,52589.0,3.163832e+08
793,0303060212 TRATAMENTO DE INSUFICIENCIA CARDIACA,0303060212,TRATAMENTO DE INSUFICIENCIA CARDIACA,2018,204836.0,3.142910e+08
1325,0303160055 TRATAMENTO DE TRANSTORNOS RELACIONA...,0303160055,TRATAMENTO DE TRANSTORNOS RELACIONADOS C/ A DU...,2018,51672.0,2.761961e+08


## 12. Conclusões

- **Escopo 2018-2024:** aplicado antes da remoção de nulos, evitando misturar o "buraco" estrutural de 2007-2012 com a ausência normal de dado dos anos maduros.
- **Nulos:** removidos após o filtro de período — dentro da janela usada, o nulo volta a significar "sem registro", tratado com a mesma regra do ISAPS/IBGE.
- **Deduplicação:** nenhuma duplicata encontrada — proteção estrutural mantida.
- **Metadados:** `arquivo_origem`/`categoria`/`metrica`/`titulo`/`descricao_variavel`/`periodo_relatorio` viraram a tabela `metadados` (3 linhas — uma por métrica levada à Silver).
- **`valor_amb` fora do escopo:** decisão explícita, não uma exclusão por qualidade.
- **Junção hospitalar:** `aih_aprovadas` e `valor_total` viraram colunas (`quantidade_aih`, `valor_total_reais`) de uma única tabela `hospitalar`, chaveada por `(procedimento_nome, ano)`.
- **Semântica do valor:** documentada duas vezes — no nome da coluna e no `description` do schema — deixando explícito que, ao contrário da `renda` do IBGE, os valores aqui são totais e podem ser somados com segurança.
- **Schema:** validado com sucesso, uma definição por tabela.